# Dependências

In [64]:
# Install required Google Cloud packages (commented out as these are typically one-time setup commands)
# !pip install gcloud
# !gcloud auth application-default login

# Import necessary Python libraries
import pandas as pd                # Data manipulation and analysis
import numpy as np                 # Numerical computing
import time                        # Time-related functions
import os                          # Operating system interfaces
import pandas_gbq                  # Pandas integration with BigQuery
from google.cloud import bigquery  # BigQuery client library
import glob                        # File path pattern matching
import openpyxl                    # Excel file handling
import csv                         # CSV file handling
import re                          # Regular expressions

# Note: The actual imports remain exactly as in the original code

# Tratamento

In [65]:
ppm18 = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\ESTADIC\\estadic_2018.xlsx', sheet_name='Política para mulheres', usecols=['Cod Uf','EPPM02','EPPM05', 'EPPM06', 'EPPM07', 'EPPM08'])
ppm18['ano'] = 2018
uf = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\ESTADIC\\estadic_2018.xlsx', sheet_name = 'Variáveis externas', usecols=[1,2,3]) # Pegando nome e codigo das UF
ppm18 = ppm18.merge(uf, right_on='COD UF',left_on='Cod Uf') # Juntando os dataframes, adicionando sigla e nome das UFs
ppm18['NOME UF'] = ppm18['NOME UF'].str.title()
ppm18= ppm18.rename(columns={'Cod Uf':'cod_uf',
                             'UF':'sigla_uf',
                             'NOME UF':'nome_uf',
                             'EPPM02':'caracterizacao_orgao_gestor',
                             'EPPM05':'genero',
                             'EPPM06':'idade',
                             'EPPM07':'cor_raca',
                             'EPPM08':'grau_instrucao'}) 
ppm18 = ppm18[['ano','cod_uf','sigla_uf','nome_uf','caracterizacao_orgao_gestor','genero','idade','cor_raca','grau_instrucao']]
ppm18

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,RO,Rondônia,Setor subordinado a outra secretaria,Feminino,29,Parda,Ensino superior completo
1,2018,12,AC,Acre,Secretaria exclusiva,Feminino,66,Parda,Mestrado
2,2018,13,AM,Amazonas,Secretaria em conjunto com outras políticas se...,Feminino,30,Branca,Especialização
3,2018,14,RR,Roraima,Setor subordinado a outra secretaria,Feminino,49,Parda,Ensino superior completo
4,2018,15,PA,Pará,Setor subordinado a outra secretaria,Feminino,55,Preta,Especialização
5,2018,16,AP,Amapá,Secretaria exclusiva,Feminino,30,Parda,Ensino superior completo
6,2018,17,TO,Tocantins,Setor subordinado a outra secretaria,Feminino,41,Branca,Especialização
7,2018,21,MA,Maranhão,Secretaria exclusiva,Feminino,63,Parda,Especialização
8,2018,22,PI,Piauí,Secretaria exclusiva,Feminino,50,Preta,Mestrado
9,2018,23,CE,Ceará,Setor subordinado a outra secretaria,Feminino,33,Branca,Ensino superior incompleto


In [66]:
ppm23 = pd.read_excel('g:\\Drives compartilhados\\República.org\\02. Áreas\\Dados e Conhecimento\\460 - Estudos e Infográficos\\07 - Mulheres no serviço público\\2026\\Dados\\bases brutas\\ESTADIC\\estadic_2023.xlsx', sheet_name='Política para Mulheres', usecols=['Cod UF','Nome UF','Sigla UF','EPPM02','EPPM05', 'EPPM06', 'EPPM07', 'EPPM08'])
ppm23['ano'] = 2023
ppm23= ppm23.rename(columns={'Cod UF':'cod_uf',
                        'Nome UF':'nome_uf',
                        'Sigla UF':'sigla_uf',
                        'EPPM02':'caracterizacao_orgao_gestor',
                        'EPPM05':'genero',
                        'EPPM06':'idade',
                        'EPPM07':'cor_raca',
                        'EPPM08':'grau_instrucao'}) 
ppm23['nome_uf'] = ppm23['nome_uf'].str.title() 
ppm23 = ppm23[['ano','cod_uf','nome_uf','sigla_uf','caracterizacao_orgao_gestor','genero','idade','cor_raca','grau_instrucao']]
ppm23

,ano,cod_uf,nome_uf,sigla_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2023,11,Rondônia,RO,Recusa,Recusa,Recusa,Recusa,Recusa
1,2023,12,Acre,AC,Secretaria estadual exclusiva,Feminino,45,Branca,Especialização
2,2023,13,Amazonas,AM,Secretaria estadual em conjunto com outras pol...,Feminino,60,Branca,Especialização
3,2023,14,Roraima,RR,Setor subordinado a outra secretaria,Feminino,57,Parda,Ensino superior completo
4,2023,15,Pará,PA,Secretaria estadual exclusiva,Feminino,39,Parda,Ensino superior completo
5,2023,16,Amapá,AP,Secretaria estadual exclusiva,Feminino,42,Parda,Especialização
6,2023,17,Tocantins,TO,Secretaria estadual exclusiva,Feminino,56,Parda,Especialização
7,2023,21,Maranhão,MA,Secretaria estadual exclusiva,Feminino,57,Parda,Ensino superior completo
8,2023,22,Piauí,PI,Secretaria estadual exclusiva,Feminino,60,Parda,Mestrado
9,2023,23,Ceará,CE,Secretaria estadual exclusiva,Feminino,38,Branca,Especialização


In [67]:
df = pd.concat([ppm18,ppm23], ignore_index=True)
df

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,RO,Rondônia,Setor subordinado a outra secretaria,Feminino,29,Parda,Ensino superior completo
1,2018,12,AC,Acre,Secretaria exclusiva,Feminino,66,Parda,Mestrado
2,2018,13,AM,Amazonas,Secretaria em conjunto com outras políticas se...,Feminino,30,Branca,Especialização
3,2018,14,RR,Roraima,Setor subordinado a outra secretaria,Feminino,49,Parda,Ensino superior completo
4,2018,15,PA,Pará,Setor subordinado a outra secretaria,Feminino,55,Preta,Especialização
5,2018,16,AP,Amapá,Secretaria exclusiva,Feminino,30,Parda,Ensino superior completo
6,2018,17,TO,Tocantins,Setor subordinado a outra secretaria,Feminino,41,Branca,Especialização
7,2018,21,MA,Maranhão,Secretaria exclusiva,Feminino,63,Parda,Especialização
8,2018,22,PI,Piauí,Secretaria exclusiva,Feminino,50,Preta,Mestrado
9,2018,23,CE,Ceará,Setor subordinado a outra secretaria,Feminino,33,Branca,Ensino superior incompleto


In [68]:
df['cor_raca'].unique()

array(['Parda', 'Branca', 'Preta', 'Recusa', 'Indígena'], dtype=object)

In [69]:
df['caracterizacao_orgao_gestor'] = df['caracterizacao_orgao_gestor'].str.title()

for col in ['caracterizacao_orgao_gestor', 'genero', 'cor_raca', 'grau_instrucao']:
    df[col] = np.where(df[col].isin(['Recusa', 'Não Informou', 'Não informou']),
                      'Sem dados',
                      df[col])

# Handle numeric age column
df['idade'] = np.where(df['idade'].isin(['Recusa', 'Não informou']),0, df['idade'])
df['idade'] = pd.to_numeric(df['idade'])
df

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao
0,2018,11,RO,Rondônia,Setor Subordinado A Outra Secretaria,Feminino,29,Parda,Ensino superior completo
1,2018,12,AC,Acre,Secretaria Exclusiva,Feminino,66,Parda,Mestrado
2,2018,13,AM,Amazonas,Secretaria Em Conjunto Com Outras Políticas Se...,Feminino,30,Branca,Especialização
3,2018,14,RR,Roraima,Setor Subordinado A Outra Secretaria,Feminino,49,Parda,Ensino superior completo
4,2018,15,PA,Pará,Setor Subordinado A Outra Secretaria,Feminino,55,Preta,Especialização
5,2018,16,AP,Amapá,Secretaria Exclusiva,Feminino,30,Parda,Ensino superior completo
6,2018,17,TO,Tocantins,Setor Subordinado A Outra Secretaria,Feminino,41,Branca,Especialização
7,2018,21,MA,Maranhão,Secretaria Exclusiva,Feminino,63,Parda,Especialização
8,2018,22,PI,Piauí,Secretaria Exclusiva,Feminino,50,Preta,Mestrado
9,2018,23,CE,Ceará,Setor Subordinado A Outra Secretaria,Feminino,33,Branca,Ensino superior incompleto


In [70]:
limites = [18, 30, 50, 65, 100]
categorias = ['Entre 18-29', 'Entre 30-49', 'Entre 50-64', 'Acima de 65']
df['faixa_etaria'] = pd.cut(df['idade'], bins=limites, labels=categorias, right=False)
df['faixa_etaria'] = df['faixa_etaria'].cat.add_categories(['Sem dados'])
df.loc[df['idade'] == 0, 'faixa_etaria'] = 'Sem dados'
df

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao,faixa_etaria
0,2018,11,RO,Rondônia,Setor Subordinado A Outra Secretaria,Feminino,29,Parda,Ensino superior completo,Entre 18-29
1,2018,12,AC,Acre,Secretaria Exclusiva,Feminino,66,Parda,Mestrado,Acima de 65
2,2018,13,AM,Amazonas,Secretaria Em Conjunto Com Outras Políticas Se...,Feminino,30,Branca,Especialização,Entre 30-49
3,2018,14,RR,Roraima,Setor Subordinado A Outra Secretaria,Feminino,49,Parda,Ensino superior completo,Entre 30-49
4,2018,15,PA,Pará,Setor Subordinado A Outra Secretaria,Feminino,55,Preta,Especialização,Entre 50-64
5,2018,16,AP,Amapá,Secretaria Exclusiva,Feminino,30,Parda,Ensino superior completo,Entre 30-49
6,2018,17,TO,Tocantins,Setor Subordinado A Outra Secretaria,Feminino,41,Branca,Especialização,Entre 30-49
7,2018,21,MA,Maranhão,Secretaria Exclusiva,Feminino,63,Parda,Especialização,Entre 50-64
8,2018,22,PI,Piauí,Secretaria Exclusiva,Feminino,50,Preta,Mestrado,Entre 50-64
9,2018,23,CE,Ceará,Setor Subordinado A Outra Secretaria,Feminino,33,Branca,Ensino superior incompleto,Entre 30-49


In [76]:
dict_esco = {
    'Ensino superior incompleto':'Até Ensino Superior Completo',
    'Ensino superior completo': 'Até Ensino Superior Completo',
    'Especialização': 'Até Especialização ou Mestrado',
    'Mestrado': 'Até Especialização ou Mestrado',
    'Doutorado': 'Até Doutorado'
}
df = df.replace({'grau_instrucao': dict_esco})
df

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao,faixa_etaria
0,2018,11,RO,Rondônia,Setor Subordinado A Outra Secretaria,Feminino,29,Parda,Até Ensino Superior Completo,Entre 18-29
1,2018,12,AC,Acre,Secretaria Exclusiva,Feminino,66,Parda,Até Especialização ou Mestrado,Acima de 65
2,2018,13,AM,Amazonas,Secretaria Em Conjunto Com Outras Políticas Se...,Feminino,30,Branca,Até Especialização ou Mestrado,Entre 30-49
3,2018,14,RR,Roraima,Setor Subordinado A Outra Secretaria,Feminino,49,Parda,Até Ensino Superior Completo,Entre 30-49
4,2018,15,PA,Pará,Setor Subordinado A Outra Secretaria,Feminino,55,Preta,Até Especialização ou Mestrado,Entre 50-64
5,2018,16,AP,Amapá,Secretaria Exclusiva,Feminino,30,Parda,Até Ensino Superior Completo,Entre 30-49
6,2018,17,TO,Tocantins,Setor Subordinado A Outra Secretaria,Feminino,41,Branca,Até Especialização ou Mestrado,Entre 30-49
7,2018,21,MA,Maranhão,Secretaria Exclusiva,Feminino,63,Parda,Até Especialização ou Mestrado,Entre 50-64
8,2018,22,PI,Piauí,Secretaria Exclusiva,Feminino,50,Preta,Até Especialização ou Mestrado,Entre 50-64
9,2018,23,CE,Ceará,Setor Subordinado A Outra Secretaria,Feminino,33,Branca,Até Ensino Superior Completo,Entre 30-49


In [ ]:
df = df[['ano',  'cod_uf', 'sigla_uf', 'nome_uf','caracterizacao_orgao_gestor',
        'genero', 'faixa_etaria', 'cor_raca', 'grau_instrucao']]

In [71]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 54 entries, 0 to 53
Data columns (total 10 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   ano                          54 non-null     int64   
 1   cod_uf                       54 non-null     int64   
 2   sigla_uf                     54 non-null     object  
 3   nome_uf                      54 non-null     object  
 4   caracterizacao_orgao_gestor  54 non-null     object  
 5   genero                       54 non-null     object  
 6   idade                        54 non-null     int64   
 7   cor_raca                     54 non-null     object  
 8   grau_instrucao               54 non-null     object  
 9   faixa_etaria                 54 non-null     category
dtypes: category(1), int64(3), object(6)
memory usage: 4.2+ KB


In [75]:
df['grau_instrucao'].unique()

array(['Ensino superior completo', 'Mestrado', 'Especialização',
       'Ensino superior incompleto', 'Doutorado', 'Sem dados'],
      dtype=object)

In [77]:
df

,ano,cod_uf,sigla_uf,nome_uf,caracterizacao_orgao_gestor,genero,idade,cor_raca,grau_instrucao,faixa_etaria
0,2018,11,RO,Rondônia,Setor Subordinado A Outra Secretaria,Feminino,29,Parda,Até Ensino Superior Completo,Entre 18-29
1,2018,12,AC,Acre,Secretaria Exclusiva,Feminino,66,Parda,Até Especialização ou Mestrado,Acima de 65
2,2018,13,AM,Amazonas,Secretaria Em Conjunto Com Outras Políticas Se...,Feminino,30,Branca,Até Especialização ou Mestrado,Entre 30-49
3,2018,14,RR,Roraima,Setor Subordinado A Outra Secretaria,Feminino,49,Parda,Até Ensino Superior Completo,Entre 30-49
4,2018,15,PA,Pará,Setor Subordinado A Outra Secretaria,Feminino,55,Preta,Até Especialização ou Mestrado,Entre 50-64
5,2018,16,AP,Amapá,Secretaria Exclusiva,Feminino,30,Parda,Até Ensino Superior Completo,Entre 30-49
6,2018,17,TO,Tocantins,Setor Subordinado A Outra Secretaria,Feminino,41,Branca,Até Especialização ou Mestrado,Entre 30-49
7,2018,21,MA,Maranhão,Secretaria Exclusiva,Feminino,63,Parda,Até Especialização ou Mestrado,Entre 50-64
8,2018,22,PI,Piauí,Secretaria Exclusiva,Feminino,50,Preta,Até Especialização ou Mestrado,Entre 50-64
9,2018,23,CE,Ceará,Setor Subordinado A Outra Secretaria,Feminino,33,Branca,Até Ensino Superior Completo,Entre 30-49


# Upload

In [78]:
# Define the BigQuery table schema with field types and descriptions
schema=[bigquery.SchemaField('ano','INTEGER',description='Ano da apuração daquele dado'),
        bigquery.SchemaField('sigla_uf','STRING',description='sigla da UF'),
        bigquery.SchemaField('nome_uf','STRING',description='nome da UF'),
        bigquery.SchemaField('cod_uf','INTEGER',description='Código do IBGE da UF'),
        bigquery.SchemaField('caracterizacao_orgao_gestor','STRING',description='Caracterização do órgão no qual o gestor está'),
        bigquery.SchemaField('genero','STRING',description='Gênero autodeclarado ou não'),
        bigquery.SchemaField('faixa_etaria','STRING',description='faixa etária da observação'),
        bigquery.SchemaField('cor_raca','STRING',description='Raça/cor da pessoa observada'),
        bigquery.SchemaField('grau_instrucao','STRING',description='Escolaridade da pessoa ou do vínculo observado com detalhamento na pós-graduação')
        ]

## Subindo para datalake
client = bigquery.Client(project='repositoriodedadosgpsp')
dataset_ref = client.dataset('cargos_lideranca')

table_ref = dataset_ref.table('ESTADIC_perfil_gestor_politica_mulheres_tipo_orgao') # nome da tabela no padrão FONTE_algo_intuitivo_dado
job_config = bigquery.LoadJobConfig(schema=schema)
job = client.load_table_from_dataframe(df, table_ref, job_config=job_config)
job.result()


LoadJob<project=repositoriodedadosgpsp, location=US, id=e373bb25-2c34-4c99-a93d-b4372bbd35b0>